# Run all pipelines

End-to-end driver for the Formulation A study. Runs, in order:

1. **Experiments** - EV/EEVS (population + sample mean), WS, DE via `run_experiments.run_all()`.
2. **Jensen gap** - ex-post Formulation B evaluation from the DE CSV (no re-solve).
3. **Figures** - executes `plotter.ipynb` to regenerate the CV / Jensen-gap map PDFs.
4. **Warm-start benchmark** (optional) - runtime-only comparison, writes `_warmstart`-suffixed files.

All outputs land under `results/` (per-scenario Gurobi logs go to `results/solver_logs/`).

> Stage 1 is slow (hundreds of MIQCP solves, can run for hours). Stages 2-3 are fast.
> Each stage is its own cell so you can re-run just the part you need.

In [1]:
import os
import sys
import time
from pathlib import Path

# --- Solver gap for this run -------------------------------------------------
# Applies to EVERY model (EV, EEVS, WS, DE). Must be set BEFORE gurobi_utils is
# imported, because DEFAULT_MIP_GAP is read at import time. The value used is
# recorded per model in summary.json's `solver_config` block.
#   1e-4  = Gurobi's default; whole pipeline ran in ~151 s.
#   1e-6  = tighter; shrinks the degenerate plateau so VSS is defensible, but DE
#           may take much longer or stall (see docs/dossier.md 15.5, 15.7, 15.8).
# Comment the line out to fall back to 1e-4.
os.environ['STOCHFAIR_MIPGAP'] = '1e-6'
# -----------------------------------------------------------------------------

SRC_DIR = str(Path.cwd().parent / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Force a fresh import of the model modules so edits in ../src take effect
# without needing a kernel restart.
import importlib
for _m in ['paths', 'gurobi_utils', 'ev_eevs_model', 'ws_model', 'de_model',
           'run_experiments', 'jensen_gap_from_de_csv', 'benchmark_warm_start']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

import gurobi_utils
import run_experiments

print('src on path:', SRC_DIR)
print('MIPGap for this run:', gurobi_utils.DEFAULT_MIP_GAP)


src on path: c:\Projects\Random personal\Stochastic fairness 2026\stochasticFairness\src
MIPGap for this run: 1e-06


## Stage 1 - Experiments (EV / EEVS / WS / DE)

Writes the per-model CSVs, `..._summary.json`, the `.txt` summary and
`scarce_<scarce>_solve_diagnostics.md` into `results/`, plus one Gurobi log per
solve into `results/solver_logs/`. **This is the long one.**

In [2]:
_t0 = time.time()
results_by_scarcity = run_experiments.run_all()
print(f'\n[stage 1] experiments finished in {time.time() - _t0:.1f}s')


scarce: 0.75

Expected value population

Expected value
Set parameter Username
Set parameter LicenseID to value 2832222
Academic license - for non-commercial use only - expires 2027-06-08
Set parameter FuncNonlinear to value 1
Set parameter FuncPieceError to value 0.0001
Set parameter DualReductions to value 0
Set parameter IntFeasTol to value 1e-07
Set parameter MIPGap to value 1e-06
Set parameter LogFile to value "C:\Projects\Random personal\Stochastic fairness 2026\stochasticFairness\results\solver_logs\EV_scarce_0.75_population_mean.log"
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11.0 (22631.2))

CPU model: Intel(R) Core(TM) Ultra 7 165H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 1563 rows, 1354 columns and 3612 nonzeros
Model fingerprint: 0xa0b9b53f
Model has 89 general constraints
Variable types: 574 continuous, 780 integer (780 binary)
Coefficient statistics:
  Mat

## Stage 2 - Jensen gap (ex-post Formulation B)

Reads the DE CSV written by stage 1 and evaluates Formulation B's
`log(E[phi**alpha])` objective on that same allocation. No Gurobi solve.

`OF_A_DE_check` should match `OF_DE` in `..._summary.json`; if it doesn't, the
DE CSV and the summary are out of sync (i.e. stale analysis).

In [3]:
import jensen_gap_from_de_csv as jg
from svi_data import SVI

csv_path = jg.find_latest_de_csv()
print(f'Reading DE solution from: {csv_path}')
jensen_result = jg.jensen_gap_from_de_csv(csv_path, svi=SVI)

print(f"Scenarios: {jensen_result['num_scenarios']}")
print(f"OF_A (DE, recomputed check): {jensen_result['OF_A_DE_check']:.6f}")
print(f"OF_B (ex-post):              {jensen_result['OF_B_expost']:.6f}")
print(f"Jensen gap:                  {jensen_result['jensen_gap']:.6f}")

Reading DE solution from: C:\Projects\Random personal\Stochastic fairness 2026\stochasticFairness\results\DE_scarce_0.75_utility_type_power_num_scenarios_150_equal_probabilities_True_nat_log_linear_pieces_length_1.csv
Scenarios: 150
OF_A (DE, recomputed check): 121.349451
OF_B (ex-post):              126.471260
Jensen gap:                  5.121808


### Freshness check

Confirms the recomputed DE objective matches the summary JSON, so the Jensen-gap
numbers really describe the run you just did (this is the exact staleness bug
caught previously - see `docs/dossier.md` Section 14.4).

In [4]:
import glob, json as _json
from paths import results_path

_summaries = sorted(glob.glob(results_path('scarce_*_summary.json')))
if _summaries:
    with open(_summaries[-1]) as f:
        _s = _json.load(f)
    _diff = abs(_s['OF_DE'] - jensen_result['OF_A_DE_check'])
    print(f"summary.json OF_DE      : {_s['OF_DE']:.6f}")
    print(f"recomputed OF_A_DE_check: {jensen_result['OF_A_DE_check']:.6f}")
    print(f"abs diff                : {_diff:.2e}")
    print('OK - analysis is fresh' if _diff < 1e-6 else 'STALE - regenerate the Jensen gap against the current DE CSV')
else:
    print('No summary JSON found - run stage 1 first.')

summary.json OF_DE      : 121.349451
recomputed OF_A_DE_check: 121.349451
abs diff                : 2.70e-13
OK - analysis is fresh


## Stage 3 - Figures

Executes `plotter.ipynb` in place, regenerating the dispersion-CV and Jensen-gap
map PDFs in `results/` against the data produced above.

In [5]:
import subprocess

_t0 = time.time()
_proc = subprocess.run(
    [sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute', '--inplace',
     'plotter.ipynb', '--ExecutePreprocessor.timeout=600'],
    cwd=str(Path.cwd()), capture_output=True, text=True)
print(_proc.stdout[-2000:])
print(_proc.stderr[-2000:])
print(f'[stage 3] plotter exit code {_proc.returncode} in {time.time() - _t0:.1f}s')


[NbConvertApp] Converting notebook plotter.ipynb to notebook
c:\Program Files\Python38\lib\site-packages\zmq\_future.py:724: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[NbConvertApp] Writing 4446670 bytes to plotter.ipynb

[stage 3] plotter exit code 0 in 81.8s


In [6]:
from paths import RESULTS_DIR

for _p in sorted(Path(RESULTS_DIR).glob('*_map.pdf')):
    print(f'{_p.name:40s} {_p.stat().st_size:>9,d} bytes  {time.ctime(_p.stat().st_mtime)}')

Gas_dispersion_cv_map.pdf                  374,462 bytes  Fri Aug 14 19:47:21 2026
Gas_network_scarcity_level_0.75_dispersion_on_map.pdf   341,545 bytes  Fri Aug 14 19:47:54 2026
Power_dispersion_cv_map.pdf                421,064 bytes  Fri Aug 14 19:47:31 2026
Power_network_scarcity_level_0.75_dispersion_on_map.pdf   341,866 bytes  Fri Aug 14 19:48:06 2026
Water_dispersion_cv_map.pdf                410,996 bytes  Fri Aug 14 19:47:41 2026
Water_network_scarcity_level_0.75_dispersion_on_map.pdf   354,143 bytes  Fri Aug 14 19:48:19 2026


## Stage 4 - Warm-start benchmark (optional)

Runtime-only experiment: re-runs EEVS (sample mean) and WS with sequential MIP
warm starts. Results are written to `_warmstart`-suffixed files so they can
**never** overwrite the baseline outputs from stage 1.

Compare `results/scarce_<scarce>_solve_diagnostics_warmstart.md` against
`results/scarce_<scarce>_solve_diagnostics.md` for the speedup.

In [7]:
from benchmark_warm_start import run_benchmark

_t0 = time.time()
run_benchmark(0.75)
print(f'\n[stage 4] warm-start benchmark finished in {time.time() - _t0:.1f}s')

[warm-start benchmark] EEVS sample-mean, scarce=0.75, warm_start_sequential=True

Expected value
Set parameter FuncNonlinear to value 1
Set parameter FuncPieceError to value 0.0001
Set parameter DualReductions to value 0
Set parameter IntFeasTol to value 1e-07
Set parameter MIPGap to value 1e-06
Set parameter LogFile to value "C:\Projects\Random personal\Stochastic fairness 2026\stochasticFairness\results\solver_logs\EV_scarce_0.75_sample_mean.log"
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11.0 (22631.2))

CPU model: Intel(R) Core(TM) Ultra 7 165H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 1563 rows, 1354 columns and 3612 nonzeros
Model fingerprint: 0x5944ed5c
Model has 89 general constraints
Variable types: 574 continuous, 780 integer (780 binary)
Coefficient statistics:
  Matrix range     [1e-01, 4e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [2e-02, 3e+02]

### Warm-start vs baseline runtime

Parses both diagnostics files and reports total Gurobi runtime per model.

In [8]:
import re
from collections import defaultdict
from paths import results_path

def _totals(md_path):
    tot = defaultdict(float)
    if not Path(md_path).exists():
        return tot
    for line in Path(md_path).read_text().splitlines():
        if not line.startswith('|'):
            continue
        cols = [c.strip() for c in line.strip('|').split('|')]
        if len(cols) < 2:
            continue
        label = cols[0]
        for c in cols:
            if re.fullmatch(r'\d+\.\d+', c):
                model = label.split('_')[0]
                tot[model] += float(c)
                break
    return tot

_base = _totals(results_path('scarce_0.75_solve_diagnostics.md'))
_warm = _totals(results_path('scarce_0.75_solve_diagnostics_warmstart.md'))

for m in sorted(set(_base) | set(_warm)):
    b, w = _base.get(m, 0.0), _warm.get(m, 0.0)
    if b and w:
        print(f'{m:8s} baseline {b:9.1f}s   warmstart {w:9.1f}s   {(b - w) / b * 100:+6.1f}%')
    else:
        print(f'{m:8s} baseline {b:9.1f}s   warmstart {w:9.1f}s')

DE       baseline      83.3s   warmstart       0.0s
EEVS     baseline      79.4s   warmstart      40.9s    +48.4%
EV       baseline       3.8s   warmstart       1.6s    +56.6%
WS       baseline     965.5s   warmstart    1024.1s     -6.1%
